In [1]:
from google.colab import drive
import sys

drive.mount('/content/drive', force_remount=True)
sys.path.append('/content/drive/MyDrive/Models/ArQModel')


ValueError: mount failed

In [ ]:
import torch
from torch.utils.data import Dataset
import h5py
import json
import os


class CaptionDataset(Dataset):
    """
    PyTorch Dataset class for image captioning using Bottom-Up & Top-Down features.
    Compatible with current training/validation pipeline.
    """

    def __init__(self, data_folder, data_name, split, transform=None):
        """
        :param data_folder: folder where data files are stored (e.g., final_dataset)
        :param data_name: base name of processed datasets (e.g., coco_5_cap_per_img_5_min_word_freq)
        :param split: one of 'TRAIN', 'VAL', or 'TEST'
        :param transform: optional image transform
        """
        self.split = split.upper()
        assert self.split in {'TRAIN', 'VAL', 'TEST'}
        self.transform = transform

        # Load the unified HDF5 features file
        self.hf = h5py.File("/content/drive/My Drive/COCO/Features/all_features.h5", 'r')
        self.features = self.hf['image_features']

        # Captions per image
        self.cpi = 5

        # Load encoded captions and their lengths
        with open(os.path.join(data_folder, f"{self.split}_CAPTIONS_{data_name}.json"), 'r') as j:
            self.captions = json.load(j)
        with open(os.path.join(data_folder, f"{self.split}_CAPLENS_{data_name}.json"), 'r') as j:
            self.caplens = json.load(j)

        # Load genome detections (object indices)
        with open(os.path.join(data_folder, f"{self.split}_GENOME_DETS_{data_name}.json"), 'r') as j:
            self.objdet = json.load(j)

        # Dataset size
        self.dataset_size = len(self.captions)

    def __getitem__(self, i):
        """Return one sample (image features + caption + caption length)."""
        objdet = self.objdet[i // self.cpi]

        # Support for both int and list formats
        img_idx = objdet if isinstance(objdet, int) else objdet[1]

        # Fetch image features
        img = torch.FloatTensor(self.features[img_idx])
        caption = torch.LongTensor(self.captions[i])
        caplen = torch.LongTensor([self.caplens[i]])


        if self.split == 'TRAIN':
            return img, caption, caplen
        else:
            all_captions = torch.LongTensor(
                self.captions[((i // self.cpi) * self.cpi):(((i // self.cpi) * self.cpi) + self.cpi)]
            )
            return img, caption, caplen, all_captions , img_idx

    def __len__(self):
        return self.dataset_size
